# SAARA QuickAPI - Complete End-to-End Example
## Dead Simple Interface for PDF → Fine-Tuned Model

Just 4 lines to configure, then use pre-built functions. Everything pre-configured and working.

```python
from saara import quickapi
quickapi.setup(model="mistral", backend="vllm")
pdf_data = quickapi.dataExtract_PDF("document.pdf")
dataset = quickapi.dataLabel_Dataset(pdf_data)
clean_data = quickapi.dataDistill_Dataset(dataset)
tokens = quickapi.dataTokenize_Dataset(clean_data)
```

## Step 1: Setup (One-Time Configuration)

In [ ]:
# For Colab: Uncomment and run
# !pip install -q vllm torch transformers

from saara import quickapi

# Configure everything in one call
config = quickapi.setup(
    model="mistral",           # or "neural-chat", "llama2", etc.
    backend="auto",             # "auto" (smart), "vllm" (fast), "ollama" (simple)
    temperature=0.7,             # Creativity: 0=fixed, 1=random
    max_tokens=2048,             # Max response length
    output_dir="./saara_results",# Where to save everything
    tokenizer="auto",            # Auto-detect from model
    verbose=True                 # Show progress
)

# Done! Everything is ready.

## Step 2: Extract Text from PDF

In [ ]:
# Create a sample PDF for demo (skip if you have your own)
from pathlib import Path

# For demo: use a text file or create a simple PDF
sample_pdf = "sample.pdf"

# If you have a real PDF, just use:
# pdf_data = quickapi.dataExtract_PDF("your_document.pdf")

# For this demo, let's show what the output looks like:
print("\n📄 Feature: PDF Extraction\n")
print("Usage: pdf_data = quickapi.dataExtract_PDF('document.pdf')\n")
print("Output structure:")
print({
    "filename": "document.pdf",
    "total_pages": 10,
    "text": "Extracted text with OCR...",
    "structured_content": ["blocks", "of", "text"],
    "metadata": {"use_ocr": True, "extraction_method": "qwen"},
    "output_file": "./saara_results/document_extracted.json"
})

## Step 3: Auto-Generate Labels (AI-Powered)

The model automatically creates Q&A pairs from your content.

In [ ]:
# Create sample text for demo
sample_text = """
    Machine learning is a subset of artificial intelligence that enables systems 
    to learn and improve from experience without being explicitly programmed. 
    It focuses on developing algorithms that can access data and learn from it 
    to make predictions or decisions. Common applications include recommendation 
    systems, image recognition, natural language processing, and autonomous vehicles.
    """

print("\n🤖 Feature: AI-Powered Dataset Labeling\n")
print("Usage: dataset = quickapi.dataLabel_Dataset(pdf_data)\n")

# Try labeling with the sample text
print("Attempting to label sample text...\n")
try:
    labeled = quickapi.dataLabel_Dataset(
        sample_text,
        label_types=["qa", "summarization"],  # Generate Q&A and summaries
        save_output=True
    )
    
    print(f"✅ Labeled {labeled['total_chunks']} chunks")
    print(f"Generated {len(labeled['labeled_items'])} training examples")
    print(f"\nSaved to: {labeled['output_file']}")
    
    # Show sample output
    if labeled['labeled_items']:
        print("\nSample labeled item:")
        sample = labeled['labeled_items'][0]
        print(f"  Instruction: {sample['instruction'][:100]}...")
        print(f"  Response: {sample['response'][:100]}...")
        print(f"  Type: {sample['label_type']}")
except Exception as e:
    print(f"ℹ️ Note: Full labeling requires inference engine. Error: {e}")
    print("\nExpected output structure:")
    print({
        "total_chunks": 5,
        "labeled_items": [
            {
                "text": "Original chunk...",
                "instruction": "What is machine learning?",
                "response": "Machine learning is...",
                "label_type": "qa"
            },
            {"text": "...", "instruction": "...", "response": "...", "label_type": "summarization"}
        ],
        "output_file": "./saara_results/labeled_dataset.jsonl"
    })

## Step 4: Distill Dataset (Quality Filtering)

Remove duplicates and low-quality items automatically.

In [ ]:
print("\n🧪 Feature: Data Distillation (Quality Filtering)\n")
print("Usage: clean_data = quickapi.dataDistill_Dataset(dataset)\n")

# Create sample labeled data for demo
sample_labeled = {
    "labeled_items": [
        {
            "text": "Sample text 1",
            "instruction": "What is AI?",
            "response": "AI is artificial intelligence...",
            "label_type": "qa"
        },
        {
            "text": "Sample text 2",
            "instruction": "What is AI?",  # Duplicate instruction
            "response": "AI is artificial intelligence...",  # Duplicate response
            "label_type": "qa"
        },
        {
            "text": "Sample text 3",
            "instruction": "Short",  # Low quality
            "response": "A",  # Too short
            "label_type": "qa"
        },
    ]
}

try:
    distilled = quickapi.dataDistill_Dataset(sample_labeled, save_output=True)
    
    print(f"✅ Distillation Complete:")
    print(f"   Original items: {distilled['original_count']}")
    print(f"   After distillation: {distilled['distilled_count']}")
    print(f"   Duplicates removed: {distilled['removed_duplicates']}")
    print(f"   Low-quality removed: {distilled['removed_low_quality']}")
    print(f"   Quality score: {distilled['quality_score']:.1%}")
    print(f"   Saved to: {distilled['output_file']}")
except Exception as e:
    print(f"Error: {e}")

## Step 5: Tokenize Dataset

Convert text to token IDs ready for training.

In [ ]:
print("\n🔢 Feature: Tokenization\n")
print("Usage: tokens = quickapi.dataTokenize_Dataset(clean_data)\n")

# Show expected tokenization output
print("Expected output after tokenization:")
print({
    "total_sequences": 100,
    "total_tokens": 102400,
    "avg_length": 1024,
    "max_length": 1024,
    "tokenizer_name": "mistral (auto-selected)",
    "tokens": "[list of token sequences]",
    "output_file": "./saara_results/tokenized_dataset.jsonl"
})

print("\n✅ Each sequence is ready for training!")

## Step 6: Format Conversion (Optional)

Convert to standard formats used by other libraries.

In [ ]:
print("\n📋 Feature: Format Conversion\n")
print("Usage: formatted = quickapi.dataConvert_Format(dataset, target_format='sharegpt')\n")

print("Supported formats:")
formats = {
    "sharegpt": {
        "use_case": "Default for modern LLM training",
        "structure": {
            "conversations": [
                {"from": "user", "value": "..."},
                {"from": "assistant", "value": "..."}
            ]
        }
    },
    "alpaca": {
        "use_case": "LLaMA fine-tuning",
        "structure": {
            "instruction": "...",
            "input": "",
            "output": "..."
        }
    },
    "openai": {
        "use_case": "OpenAI API compatibility",
        "structure": {
            "messages": [
                {"role": "user", "content": "..."},
                {"role": "assistant", "content": "..."}
            ]
        }
    }
}

for fmt, details in formats.items():
    print(f"\n  {fmt.upper()}")
    print(f"    Use: {details['use_case']}")
    print(f"    Structure: {details['structure']}")

## Complete Pipeline Example

Here's how to use all functions together in production:

In [ ]:
print("\n" + "="*60)
print("COMPLETE PRODUCTION PIPELINE")
print("="*60 + "\n")

production_code = '''
from saara import quickapi

# 1️⃣ Setup (one time)
quickapi.setup(
    model="mistral",
    backend="vllm",
    output_dir="./results"
)

# 2️⃣ Extract PDF
pdf_data = quickapi.dataExtract_PDF(
    "research_paper.pdf",
    use_ocr=True  # For complex layouts
)

# 3️⃣ Generate labels (AI-powered)
dataset = quickapi.dataLabel_Dataset(
    pdf_data,
    label_types=["qa", "summarization", "classification"]
)

# 4️⃣ Clean dataset (remove duplicates & low quality)
clean_data = quickapi.dataDistill_Dataset(
    dataset,
    min_quality=0.7,
    remove_duplicates=True
)

# 5️⃣ Tokenize for training
tokens = quickapi.dataTokenize_Dataset(
    clean_data,
    max_length=1024
)

# 6️⃣ Format for training framework
formatted = quickapi.dataConvert_Format(
    tokens,
    target_format="sharegpt"
)

# 7️⃣ Now use formatted["output_file"] with any training library:
# - Hugging Face: SFTTrainer
# - PEFT: QLoRA
# - vLLM: API-based fine-tuning
# - Ollama: Local fine-tuning

print(f"✅ Dataset ready at: {formatted['output_file']}")
'''

print(production_code)

## Advanced: Check Status Anytime

In [ ]:
import json

status = quickapi.get_status()
print("\n📊 Current Configuration:")
print(json.dumps(status, indent=2))

## What's Happening Under the Hood

| Function | What It Does | Output |
|----------|------|-------|
| `setup()` | Configure model, backend, tokenizer | Config dict |
| `dataExtract_PDF()` | Extract text + OCR for complex layouts | Text + metadata |
| `dataLabel_Dataset()` | Use vLLM/Ollama to generate Q&A pairs | JSONL with instruction-response |
| `dataDistill_Dataset()` | Remove duplicates + low quality items | Cleaned JSONL |
| `dataTokenize_Dataset()` | Convert text → token IDs | Tokenized sequences |
| `dataConvert_Format()` | Convert to training framework format | ShareGPT/Alpaca/OpenAI JSONL |

## Key Features

✅ **Everything Pre-Configured**
- Tokenizers auto-loaded
- Output formats standardized
- Error handling built-in

✅ **No API Keys Needed**
- Uses local vLLM or Ollama
- Data stays on your machine
- Fully private

✅ **Works Locally & Cloud**
- Local machine: vLLM (fast)
- Google Colab: vLLM (one-shot!)
- Kaggle: vLLM native support
- Docker: Containerized

✅ **Production Ready**
- Fallback mechanisms
- Quality metrics
- Automatic logging

## Troubleshooting

### "ModuleNotFoundError: No module named 'vllm'"
**Solution**: `pip install vllm`

### "Inference engine not initialized"
**Solution**: Call `quickapi.setup()` before using other functions

### "Ollama not responding"
**Solution**: Ensure `ollama serve` is running in a terminal

### "Out of memory"
**Solution**: Use smaller model: `quickapi.setup(model="neural-chat")`

### "PDF extraction failed"
**Solution**: Try with `use_ocr=False` for simple PDFs, or install OCR models manually